# Phase 8 — ELECTRA-small Fine-tuning

Fine-tuning του `google/electra-small-discriminator` για Food Hazard Detection.

**Γιατί ELECTRA-small:**
- Εκπαιδεύεται ως discriminator (replaced token detection) — πιο αποδοτικό από MLM
- Συχνά καλύτερο από DistilBERT με λιγότερα parameters
- Έχει κανονικό classification head, χωρίς προβλήματα φόρτωσης
- Τρέχει γρήγορα στο Colab T4

**Στρατηγική:** train-only + early stopping (max 15 epochs)

In [ ]:
!pip install sentencepiece -q

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import copy
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_train = make_text(train)
texts_valid = make_text(valid)
texts_test  = make_text(test)

print(f'Train: {len(texts_train)}')
print(f'Valid: {len(texts_valid)}')
print(f'Test:  {len(texts_test)}')

In [ ]:
# Label Encoding
le_hazard  = LabelEncoder()
le_product = LabelEncoder()

all_hazard  = pd.concat([train['hazard-category'],  valid['hazard-category']])
all_product = pd.concat([train['product-category'], valid['product-category']])
le_hazard.fit(all_hazard)
le_product.fit(all_product)

y_train_hazard  = le_hazard.transform(train['hazard-category'])
y_train_product = le_product.transform(train['product-category'])
y_valid_hazard  = le_hazard.transform(valid['hazard-category'])
y_valid_product = le_product.transform(valid['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Hazard classes:  {n_hazard}')
print(f'Product classes: {n_product}')

In [ ]:
MODEL_NAME   = 'google/electra-small-discriminator'
BATCH_SIZE   = 32   # ELECTRA-small είναι μικρό, αντέχει batch_size=32
MAX_LENGTH   = 128
LR           = 3e-5  # ELECTRA συνήθως θέλει λίγο μεγαλύτερο LR από BERT
MAX_EPOCHS   = 15
PATIENCE     = 3
WARMUP_RATIO = 0.1

print(f'Φόρτωση tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded!')

dummy_labels = np.zeros(len(texts_test), dtype=int)

In [ ]:
class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        labels = batch.pop('label').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()
    return total_loss / len(loader)


def get_predictions(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            preds = model(**batch).logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)


def get_probabilities(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            probs = torch.softmax(model(**batch).logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)


def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product, verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    mask = (np.array(y_true_hazard) == np.array(y_pred_hazard))
    f1_product = f1_score(
        np.array(y_true_product)[mask],
        np.array(y_pred_product)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'Σωστά hazard predictions:           {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

In [ ]:
# DataLoaders
train_loader_hazard  = DataLoader(FoodHazardDataset(texts_train, y_train_hazard,  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
train_loader_product = DataLoader(FoodHazardDataset(texts_train, y_train_product, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
valid_loader_hazard  = DataLoader(FoodHazardDataset(texts_valid, y_valid_hazard,  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
valid_loader_product = DataLoader(FoodHazardDataset(texts_valid, y_valid_product, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader_hazard   = DataLoader(FoodHazardDataset(texts_test,  dummy_labels,    tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader_product  = DataLoader(FoodHazardDataset(texts_test,  dummy_labels,    tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader_hazard)}')
print(f'Batch size: {BATCH_SIZE} | Max epochs: {MAX_EPOCHS} | LR: {LR}')

## Εκπαίδευση ELECTRA — Hazard Classifier

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ ELECTRA-small — HAZARD ===')
print(f'Train: {len(texts_train)} | Valid: {len(texts_valid)} | Max epochs: {MAX_EPOCHS}\n')

electra_hazard = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_hazard
).to(device)

optimizer_hazard = AdamW(electra_hazard.parameters(), lr=LR, weight_decay=0.01)

total_steps_hazard  = len(train_loader_hazard) * MAX_EPOCHS
warmup_steps_hazard = int(total_steps_hazard * WARMUP_RATIO)
scheduler_hazard = get_linear_schedule_with_warmup(
    optimizer_hazard,
    num_warmup_steps=warmup_steps_hazard,
    num_training_steps=total_steps_hazard
)
print(f'Total steps: {total_steps_hazard} | Warmup steps: {warmup_steps_hazard}\n')

best_f1_hazard    = 0
best_state_hazard = None
patience_counter  = 0

for epoch in range(MAX_EPOCHS):
    loss  = train_epoch(electra_hazard, train_loader_hazard, optimizer_hazard, scheduler_hazard)
    preds = le_hazard.inverse_transform(get_predictions(electra_hazard, valid_loader_hazard))
    f1    = f1_score(valid['hazard-category'], preds, average='macro', zero_division=0)

    print(f'Epoch {epoch+1}/{MAX_EPOCHS} | Loss: {loss:.4f} | Valid F1 Hazard: {f1:.4f}', end='')

    if f1 > best_f1_hazard:
        best_f1_hazard    = f1
        best_state_hazard = copy.deepcopy(electra_hazard.state_dict())
        patience_counter  = 0
        print(' αποθηκεύτηκε')
    else:
        patience_counter += 1
        print(f' (patience {patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping στο epoch {epoch+1}!')
            break

electra_hazard.load_state_dict(best_state_hazard)
print(f'\nΚαλύτερο F1 Hazard: {best_f1_hazard:.4f}')

## Εκπαίδευση ELECTRA — Product Classifier

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ ELECTRA-small — PRODUCT ===')
print(f'Train: {len(texts_train)} | Valid: {len(texts_valid)} | Max epochs: {MAX_EPOCHS}\n')

electra_product = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_product
).to(device)

optimizer_product = AdamW(electra_product.parameters(), lr=LR, weight_decay=0.01)

total_steps_product  = len(train_loader_product) * MAX_EPOCHS
warmup_steps_product = int(total_steps_product * WARMUP_RATIO)
scheduler_product = get_linear_schedule_with_warmup(
    optimizer_product,
    num_warmup_steps=warmup_steps_product,
    num_training_steps=total_steps_product
)
print(f'Total steps: {total_steps_product} | Warmup steps: {warmup_steps_product}\n')

best_f1_product    = 0
best_state_product = None
patience_counter   = 0

for epoch in range(MAX_EPOCHS):
    loss  = train_epoch(electra_product, train_loader_product, optimizer_product, scheduler_product)
    preds = le_product.inverse_transform(get_predictions(electra_product, valid_loader_product))
    f1    = f1_score(valid['product-category'], preds, average='macro', zero_division=0)

    print(f'Epoch {epoch+1}/{MAX_EPOCHS} | Loss: {loss:.4f} | Valid F1 Product: {f1:.4f}', end='')

    if f1 > best_f1_product:
        best_f1_product    = f1
        best_state_product = copy.deepcopy(electra_product.state_dict())
        patience_counter   = 0
        print(' αποθηκεύτηκε')
    else:
        patience_counter += 1
        print(f' (patience {patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping στο epoch {epoch+1}!')
            break

electra_product.load_state_dict(best_state_product)
print(f'\nΚαλύτερο F1 Product: {best_f1_product:.4f}')

## Αξιολόγηση στο Validation Set

In [ ]:
preds_hazard  = le_hazard.inverse_transform(get_predictions(electra_hazard,  valid_loader_hazard))
preds_product = le_product.inverse_transform(get_predictions(electra_product, valid_loader_product))

print('=== ΑΞΙΟΛΟΓΗΣΗ ELECTRA-small (validation) ===\n')
score_electra = official_st1_score(
    valid['hazard-category'],  preds_hazard,
    valid['product-category'], preds_product
)

print('\n=== ΣΥΓΚΡΙΣΗ (validation scores) ===')
print(f'TF-IDF + SVM:              0.7419')
print(f'Fine-tuned DistilBERT:     0.7998  (Kaggle: 0.723)')
print(f'BERT+SVM Ensemble:         0.8028  (Kaggle: 0.755) ← best Kaggle')
print(f'DeBERTa (nli-v3-small):    0.7229')
print(f'ELECTRA-small:             {score_electra:.4f}')

## Αποθήκευση Probabilities για Ensemble

In [ ]:
# Test predictions
test_hazard  = le_hazard.inverse_transform(get_predictions(electra_hazard,  test_loader_hazard))
test_product = le_product.inverse_transform(get_predictions(electra_product, test_loader_product))

# Submission ELECTRA μόνο
pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
}).to_csv('submission_electra.csv', index=False)
print('Αποθηκεύτηκε: submission_electra.csv')

# Probabilities για ensemble με SVM
np.save('electra_valid_hazard_probs.npy',  get_probabilities(electra_hazard,  valid_loader_hazard))
np.save('electra_valid_product_probs.npy', get_probabilities(electra_product, valid_loader_product))
np.save('electra_test_hazard_probs.npy',   get_probabilities(electra_hazard,  test_loader_hazard))
np.save('electra_test_product_probs.npy',  get_probabilities(electra_product, test_loader_product))

print('\nΈτοιμα για ensemble:')
print('  electra_valid_hazard_probs.npy')
print('  electra_valid_product_probs.npy')
print('  electra_test_hazard_probs.npy')
print('  electra_test_product_probs.npy')